# Pathway coverage after GO:BP-prior retraining

CLAMPfull is retrained with the pinned GO:BP prior. Coverage is evaluated independently against Reactome, canonical non-Reactome pathways, and CellMarker using `clusterProfiler::enricher`. Each compendium uses its own explicit model-gene universe and its own eligible-pathway denominator.

In [ ]:
suppressPackageStartupMessages({
    library(data.table)
    library(ggplot2)
    library(here)
})

COVERAGE_DIR <- here(snakemake@params[["coverage_dir"]])
coverage <- fread(snakemake@input[["coverage_long"]])
cross <- fread(snakemake@input[["cross_dataset"]])
panel <- fread(snakemake@input[["panel_ready"]])
stopifnot(nrow(coverage) > 0, nrow(cross) > 0, nrow(panel) > 0)
coverage[, .N, by = .(dataset, model, database)]

## Dataset-specific ORA universes and denominators

In [ ]:
coverage[, .(
    universe_size = unique(universe_size),
    eligible_pathways = unique(eligible_pathways),
    top_one_percent_genes = unique(query_target)
), by = .(dataset, database_label)]

## ARCHS4 coverage by sample fraction

In [ ]:
knitr::include_graphics(snakemake@input[["arch_png"]])

The boxes summarize the three seeds. Points retain the individual runs, and the thin dashed trajectory connects seed means. CLAMPbase is a deterministic reference for fixed inputs. Labels show the mean recovered count followed by the percentage of eligible pathways.

At 100% there is no subsampling, so all three seeds share one SVD and one CLAMPbase fit. The CLAMPbase reference mark at that fraction has no seed-to-seed spread by construction; only CLAMPfull varies there.

## Full-data comparison

In [ ]:
knitr::include_graphics(snakemake@input[["cross_png"]])

Percentages in the cross-dataset panel use dataset-specific universes and therefore dataset-specific eligible-pathway denominators; the source tables retain both the counts and denominators.

The three universes differ enough that the panel's absolute counts are **not** a like-for-like comparison. ARCHS4 scores over 18,423 genes and GTEx over 21,613, but recount2 enters the repo already filtered to 6,000, which also shrinks its eligible-pathway denominators (1,148 Reactome against 1,370 and 1,381). Restricting each ORA to the genes its own model could have recovered is the correct construction, but part of recount2's lower recovered count is denominator rather than model quality. Compare the percentages first; the table above gives the denominators behind every count.